# CWSB — Retrain the model on new data (one notebook, run top to bottom)

Use this whenever new **whole-plant** photos arrive (healthy and/or infected).
It rebuilds the same model we already have (`contender_B`, ConvNeXt-Tiny, 384 px)
on **all the old photos plus the new ones**, and saves it as a new version
(`v2`, `v3`, ...). It never overwrites an older version.

## Before you run

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Put these in one Google Drive folder (default `MyDrive/cwsb/`):

```
MyDrive/cwsb/
├── Data_final.zip          <- the original 498 photos (always keep this in the list)
├── batch_2026_12.zip       <- each new delivery, as its own zip
├── split_v1.csv            <- from the repo: splits/split_v1.csv
└── v1/contender_B.pt       <- the current model (optional, used for comparison)
```

3. **How a data zip must look inside.** Two folders, any spelling that starts with
   `Healthy` and `Unhealthy` / `Infected`:

```
batch_2026_12.zip
├── Healthy/
│   ├── IMG_0001.jpg                 <- loose photos are fine
│   └── plant_17/IMG_0002.jpg        <- better: one sub-folder per plant
└── Unhealthy/
    └── plant_31/IMG_0101.jpg
```

   If photos are in **one sub-folder per plant**, the notebook uses that as the plant
   identity. If they are loose, it guesses plants from consecutive file numbers, as v1 did.

4. Edit the **CONFIG** cell (usually only `VERSION` and `DATA_SOURCES`), then
   **Runtime → Run all**.

## What it does

| Step | What happens |
|---|---|
| 1 | Unzips every data source, finds the Healthy / Unhealthy photos |
| 2 | Cleans: fixes rotation, strips GPS, checks blur / glare / darkness, removes duplicates |
| 3 | Splits **by plant**. Photos that were in the previous split keep their place, so the old test set stays the same and versions can be compared |
| 4 | Extra crops of healthy training photos (only while healthy photos are scarce) |
| 5 | Trains contender B |
| 6 | Sets thresholds from the **healthy** validation scores (never from infected quantiles) |
| 7 | Tests on the held-out plants, compares with the previous model, saves everything to Drive |

**Only whole-plant photos from 2–3 m belong here.** Stem close-ups need a different
(detection) model; mixing them in will make this model worse.

In [ ]:
# ============================== CONFIG ==============================
VERSION = "v2"                      # name of the model this run produces: v2, v3, ...

DRIVE_DIR = "/content/drive/MyDrive/cwsb"

# Every zip (or folder) of photos to train on. Paths are relative to DRIVE_DIR.
# ALWAYS keep the earlier sources and ADD new deliveries at the end.
DATA_SOURCES = [
    "Data_final.zip",
    # "batch_2026_12.zip",
]

# The split file of the previous version (keeps the old train/val/test plants fixed).
# Use None only for a brand-new project with no earlier model.
PREVIOUS_SPLIT = "split_v1.csv"

# The previous model, scored on the same test set for a side-by-side comparison.
# Use None to skip the comparison.
PREVIOUS_MODEL = "v1/contender_B.pt"

# "imagenet" = train fresh on old + new data (recommended).
# "previous" = start from PREVIOUS_MODEL's weights and keep training on old + new data.
START_FROM = "imagenet"

# --- things you normally leave alone ---
TILE_HEALTHY = "auto"       # extra crops of healthy training photos: True / False / "auto"
TILE_AUTO_BELOW = 150       # "auto" switches tiling off once this many healthy training photos exist
EPOCHS = 40
PATIENCE = 6
BATCH = 16
TRAIN_SHORT_SIDE = 768      # photos are stored at this size for training (v1 did the same)
FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
MIN_HEALTHY_VAL_FOR_AUTO_THRESHOLDS = 20
OVERWRITE = False           # True lets a run replace an existing VERSION folder
ALLOW_CPU = False           # True only for a quick smoke test without a GPU
WORK_DIR = "/content/cwsb_work"

In [ ]:
# ---------- Setup: GPU check, Drive, libraries ----------
import csv, hashlib, json, random, re, shutil, subprocess, sys, time, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path


def _need(module, pip_name=None):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=False)


_need("timm"); _need("cv2", "opencv-python-headless"); _need("pillow_heif", "pillow-heif")

import cv2
import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except Exception:
    print("note: HEIC support not available; .heic photos will be skipped")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU")
if DEVICE != "cuda" and not ALLOW_CPU:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again.")

SEED = 42
IMG = 384
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k_384"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

DRIVE = Path(DRIVE_DIR)
WORK = Path(WORK_DIR)
RAW, CLEAN, TILES = WORK / "raw", WORK / "clean", WORK / "tiles"
OUT = DRIVE / VERSION


def _drive_path(p):
    return None if p is None else (Path(p) if Path(p).is_absolute() else DRIVE / p)


PREV_SPLIT_PATH = _drive_path(PREVIOUS_SPLIT)
PREV_MODEL_PATH = _drive_path(PREVIOUS_MODEL)

# ---- checks that stop a wasted run early ----
problems = []
if not re.fullmatch(r"v\d+[\w.-]*", VERSION):
    problems.append(f'VERSION "{VERSION}" should look like v2, v3, ...')
if (OUT / "contender_B.pt").exists() and not OVERWRITE:
    problems.append(f"{OUT} already holds a model. Pick a new VERSION (or set OVERWRITE = True).")
for s in DATA_SOURCES:
    if not _drive_path(s).exists():
        problems.append(f"data source not found: {_drive_path(s)}")
if not DATA_SOURCES:
    problems.append("DATA_SOURCES is empty")
if PREV_SPLIT_PATH and not PREV_SPLIT_PATH.exists():
    problems.append(f"PREVIOUS_SPLIT not found: {PREV_SPLIT_PATH}")
if PREV_MODEL_PATH and not PREV_MODEL_PATH.exists():
    problems.append(f"PREVIOUS_MODEL not found: {PREV_MODEL_PATH}")
if START_FROM not in ("imagenet", "previous"):
    problems.append('START_FROM must be "imagenet" or "previous"')
if START_FROM == "previous" and not (PREV_MODEL_PATH and PREV_SPLIT_PATH):
    problems.append('START_FROM = "previous" needs both PREVIOUS_MODEL and PREVIOUS_SPLIT '
                    "(without the old split, photos the old model trained on could land in the new test set)")
if problems:
    raise SystemExit("Fix the CONFIG cell:\n  - " + "\n  - ".join(problems))

for d in (RAW, CLEAN, TILES):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
OUT.mkdir(parents=True, exist_ok=True)
print(f"OK. This run will write model {VERSION} to {OUT}")

## Step 1 — Unpack the data and find the photos

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic", ".heif"}


def class_of(folder_name):
    n = folder_name.strip().lower()
    if n.startswith(("unhealthy", "infected", "wsb", "diseased")):
        return "Unhealthy"
    if n.startswith("healthy"):
        return "Healthy"
    return None


found, ignored = [], Counter()
for si, s in enumerate(DATA_SOURCES):
    src = _drive_path(s)
    tag = re.sub(r"[^\w-]+", "_", src.stem)
    if src.is_dir():
        root = src
    else:
        root = RAW / f"{si:02d}_{tag}"
        with zipfile.ZipFile(src) as z:
            z.extractall(root)
    n0 = len(found)
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXT or p.name.startswith("._"):
            continue
        parts = p.relative_to(root).parts[:-1]
        hit = [(i, class_of(x)) for i, x in enumerate(parts) if class_of(x)]
        if not hit:
            ignored[tag] += 1
            continue
        i, label = hit[-1]                       # nearest class folder above the photo
        plant_dir = "/".join(parts[i + 1:])      # sub-folder(s) under the class folder = the plant
        found.append({"path": p, "label": label, "source": tag,
                      "plant_hint": f"{tag}/{label}/{plant_dir}" if plant_dir else ""})
    c = Counter(r["label"] for r in found[n0:])
    print(f"{src.name}: {c.get('Healthy', 0)} healthy, {c.get('Unhealthy', 0)} unhealthy"
          + (f", {ignored[tag]} images ignored (not under a Healthy/Unhealthy folder)" if ignored[tag] else ""))
    if not c:
        raise SystemExit(f"No photos found in {src.name}. It must contain Healthy/ and Unhealthy/ folders.")

print(f"\nTotal found: {len(found)}")

## Step 2 — Clean and check quality

Same rules as v1 (`src/clean.py`): rotation fixed, all metadata (GPS) removed,
blur / glare / darkness measured, exact and near duplicates found.
Blurry, burnt-out or dark photos are **excluded** from training and testing.

In [ ]:
BLUR_MIN, CLIPPED_MAX, DARK_MAX, NEAR_DUP_HAMMING = 60.0, 0.20, 0.40, 6


def dhash(gray, size=8):
    small = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = small[:, 1:] > small[:, :-1]
    return int("".join("1" if b else "0" for b in bits.flatten()), 2)


def shrink(img, short=TRAIN_SHORT_SIDE):
    s = short / min(img.size)
    return img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS) if s < 1 else img


def clean_one(r):
    try:
        raw = r["path"].read_bytes()
        img = ImageOps.exif_transpose(Image.open(r["path"])).convert("RGB")
    except Exception as e:
        return {"error": f"{r['path'].name}: {e}"}
    g_full = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2GRAY)
    scale = 1024 / max(g_full.shape[1], 1)
    g = cv2.resize(g_full, (1024, max(int(g_full.shape[0] * scale), 1)))
    blur = float(cv2.Laplacian(g, cv2.CV_64F).var())
    clipped, dark = float((g > 250).mean()), float((g < 15).mean())
    flags = [n for n, bad in (("blurry", blur < BLUR_MIN), ("clipped", clipped > CLIPPED_MAX),
                              ("dark", dark > DARK_MAX)) if bad]
    return {"label": r["label"], "source": r["source"], "plant_hint": r["plant_hint"],
            "orig": str(r["path"]), "stem": r["path"].stem, "small": shrink(img),
            "width": img.width, "height": img.height, "blur": round(blur, 2),
            "clipped": round(clipped, 4), "dark": round(dark, 4),
            "src_sha256": hashlib.sha256(raw).hexdigest(), "dhash": f"{dhash(g):016x}",
            "flags": "|".join(flags)}


t0 = time.time()
rows, errors, by_sha, used_names = [], [], {}, set()
exact_dups, label_conflicts = [], []
for d in ("Healthy", "Unhealthy"):
    (CLEAN / d).mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(4) as ex:
    for k, c in enumerate(ex.map(clean_one, found)):
        if "error" in c:
            errors.append(c["error"])
            continue
        small = c.pop("small")
        stem = c.pop("stem")
        if c["src_sha256"] in by_sha:                     # the very same file again
            first = by_sha[c["src_sha256"]]
            (label_conflicts if first["label"] != c["label"] else exact_dups).append((first["file"], c["orig"]))
            continue
        name = f"{c['label']}/{stem}.jpg"
        if name.lower() in used_names:                    # same file name, different photo
            name = f"{c['label']}/{stem}__{c['src_sha256'][:8]}.jpg"
        used_names.add(name.lower())
        small.save(CLEAN / name, "JPEG", quality=90)      # no EXIF/GPS is written
        c["file"] = name
        by_sha[c["src_sha256"]] = c
        rows.append(c)
        if (k + 1) % 100 == 0:
            print(f"  {k + 1}/{len(found)}  ({time.time() - t0:.0f}s)")

# a photo filed under BOTH classes is a labelling error: drop it everywhere
conflict_files = {a for a, _ in label_conflicts}
for r in rows:
    if r["file"] in conflict_files:
        r["flags"] = (r["flags"] + "|" if r["flags"] else "") + "label_conflict"

# near duplicates (same scene, different file)
near = []
hashes = [int(r["dhash"], 16) for r in rows]
for i in range(len(rows)):
    hi = hashes[i]
    for j in range(i + 1, len(rows)):
        d = bin(hi ^ hashes[j]).count("1")
        if d <= NEAR_DUP_HAMMING:
            near.append((i, j, d))

cols = ["file", "label", "source", "plant_hint", "width", "height", "blur", "clipped", "dark",
        "src_sha256", "dhash", "flags"]
with open(OUT / "images.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader(); w.writerows(rows)
with open(OUT / "duplicates.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["file_a", "file_b", "kind", "hamming"])
    w.writerows([(a, b, "exact_skipped", 0) for a, b in exact_dups])
    w.writerows([(a, b, "LABEL_CONFLICT", 0) for a, b in label_conflicts])
    w.writerows([(rows[i]["file"], rows[j]["file"], "near", d) for i, j, d in near])

flagged = [r for r in rows if r["flags"]]
cross_near = [(i, j) for i, j, _ in near if rows[i]["label"] != rows[j]["label"]]
print(f"\nCleaned {len(rows)} photos in {time.time() - t0:.0f}s")
print(f"  unreadable files skipped : {len(errors)}")
for e in errors[:10]:
    print("     ", e)
print(f"  exact duplicates skipped : {len(exact_dups)}")
print(f"  excluded for quality     : {len(flagged)}  " + str(dict(Counter(x for r in flagged for x in r['flags'].split('|')))))
print(f"  near-duplicate pairs     : {len(near)}")
if label_conflicts:
    print(f"\n  !! {len(label_conflicts)} photo(s) appear under BOTH Healthy and Unhealthy - dropped. Fix the labels:")
    for a, b in label_conflicts[:20]:
        print("     ", a, "<->", b)
if cross_near:
    print(f"\n  !! {len(cross_near)} near-identical pair(s) carry different labels - please review:")
    for i, j in cross_near[:20]:
        print("     ", rows[i]["file"], "<->", rows[j]["file"])

## Step 3 — Split by plant (old plants stay where they were)

- A plant never appears in two of train / validation / test.
- Every photo that was in the previous split keeps its plant and its place. So the
  old test photos are still test photos, and old vs new model can be compared fairly.
- New photos are grouped into plants (sub-folder if given, otherwise consecutive file
  numbers), and whole plants are dealt out to reach 70 / 15 / 15 overall.

In [ ]:
FRAME_GAP, TIME_GAP = 4, 120.0
TS_RE = re.compile(r"(?P<pre>.*IMG)_(?P<d>\d{8})_(?P<t>\d{6})", re.I)
NUM_RE = re.compile(r"^(?P<pre>.*?)(?P<num>\d+)(?P<post>[^\d]*)$")


def sort_key(r):
    stem = Path(r["file"]).stem.split("__")[0]
    base = (r["label"], r["source"])
    m = TS_RE.match(stem)
    if m:
        try:
            ts = datetime.strptime(m.group("d") + m.group("t"), "%Y%m%d%H%M%S").timestamp()
            return (base, "ts:" + m.group("pre"), ts)
        except ValueError:
            pass
    m = NUM_RE.match(stem)
    if m:
        return (base, "fr:" + m.group("pre") + "|" + m.group("post"), float(m.group("num")))
    return (base, "one:" + stem, 0.0)


# ---- 3a. photos already placed by the previous split ----
prev_by_sha, prev_by_file, prev_rows = {}, {}, []
if PREV_SPLIT_PATH:
    with open(PREV_SPLIT_PATH, newline="") as f:
        prev_rows = list(csv.DictReader(f))
    for p in prev_rows:
        if p.get("src_sha256"):
            prev_by_sha[p["src_sha256"]] = p
        prev_by_file[p["file"].lower()] = p

matched = set()
for r in rows:
    p = prev_by_sha.get(r["src_sha256"]) or prev_by_file.get(r["file"].lower())
    if p:
        r["group"], r["split"], r["old"] = p["group"], p["split"], True
        matched.add(p["file"])
        if p["label"] != r["label"]:
            print(f"  note: {r['file']} was {p['label']} before, is {r['label']} now (kept in {p['split']})")
    else:
        r["old"] = False

missing_old = [p["file"] for p in prev_rows if p["file"] not in matched]
if missing_old:
    raise SystemExit(
        f"{len(missing_old)} photo(s) from the previous split are missing from DATA_SOURCES, e.g. "
        f"{missing_old[:5]}.\nKeep the earlier zips in DATA_SOURCES - the model must be retrained on old + new data.")

old_rows = [r for r in rows if r["old"]]
new_rows = [r for r in rows if not r["old"]]
print(f"photos carried over from the previous split: {len(old_rows)}   new photos: {len(new_rows)}")

# ---- 3b. group the NEW photos into plants ----
parent = list(range(len(new_rows)))


def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


def union(a, b):
    parent[find(a)] = find(b)


by_hint = defaultdict(list)
for i, r in enumerate(new_rows):
    if r["plant_hint"]:
        by_hint[r["plant_hint"]].append(i)
for members in by_hint.values():                      # explicit plant folders
    for i in members[1:]:
        union(members[0], i)

loose = sorted((i for i, r in enumerate(new_rows) if not r["plant_hint"]), key=lambda i: sort_key(new_rows[i]))
prev = None
for i in loose:                                       # consecutive frames / close timestamps
    base, series, pos = sort_key(new_rows[i])
    gap = TIME_GAP if series.startswith("ts:") else FRAME_GAP
    if prev and prev[0] == (base, series) and pos - prev[1] <= gap:
        union(prev[2], i)
    prev = ((base, series), pos, i)

# near-duplicates must never straddle two splits
idx_new = {id(r): i for i, r in enumerate(new_rows)}
pinned = defaultdict(set)                             # new plant -> split(s) of old look-alikes
pending = []
for i, j, _ in near:
    a, b = rows[i], rows[j]
    if not a["old"] and not b["old"]:
        union(idx_new[id(a)], idx_new[id(b)])
    elif a["old"] != b["old"]:
        o, n = (a, b) if a["old"] else (b, a)
        pending.append((idx_new[id(n)], o))
for ni, o in pending:
    if o["split"] != "excluded":
        pinned[find(ni)].add((o["split"], o["group"]))

groups = defaultdict(list)
for i, r in enumerate(new_rows):
    groups[find(i)].append(r)

old_ids = [int(m.group(1)) for r in old_rows if (m := re.fullmatch(r"plant_(\d+)", r["group"]))]
next_id = max(old_ids, default=-1) + 1

# ---- 3c. deal new plants out to train / val / test ----
rng = random.Random(SEED)
got = {lab: Counter() for lab in ("Healthy", "Unhealthy")}
for r in old_rows:
    if r["split"] in FRACTIONS and not r["flags"]:
        got[r["label"]][r["split"]] += 1

free = defaultdict(list)
n_pinned = n_conflict = 0
for root_i in sorted(groups):
    g = groups[root_i]
    gid = f"plant_{next_id:04d}"
    next_id += 1
    usable = [r for r in g if not r["flags"]]
    pins = pinned.get(root_i, set())
    pin_splits = {s for s, _ in pins}
    for r in g:
        r["group"] = gid
    if len(pin_splits) > 1 or len({r["label"] for r in g}) > 1:
        for r in g:                                   # looks like plants in two different sets, or mixed labels
            r["split"] = "excluded"
        n_conflict += 1
    elif pin_splits:
        s = next(iter(pin_splits))
        for r in g:
            r["split"] = s
        got[g[0]["label"]][s] += len(usable)
        n_pinned += 1
    elif usable:
        free[g[0]["label"]].append(g)
    else:
        for r in g:
            r["split"] = "excluded"

for label, gs in free.items():
    rng.shuffle(gs)
    total = sum(got[label].values()) + sum(len([r for r in g if not r["flags"]]) for g in gs)
    want = {s: total * frac for s, frac in FRACTIONS.items()}
    for s in ("test", "val", "train"):                # make sure no set is left without this class
        if got[label][s] == 0 and gs:
            g = gs.pop()
            for r in g:
                r["split"] = s
            got[label][s] += len([r for r in g if not r["flags"]])
    for g in gs:
        s = max(FRACTIONS, key=lambda s: (want[s] - got[label][s]) / want[s])
        for r in g:
            r["split"] = s
        got[label][s] += len([r for r in g if not r["flags"]])

for r in rows:
    if r["flags"]:
        r["split"] = "excluded"

# ---- 3d. checks ----
plant_splits = defaultdict(set)
for r in rows:
    if r["split"] != "excluded":
        plant_splits[r["group"]].add(r["split"])
leaks = {g: s for g, s in plant_splits.items() if len(s) > 1}
if leaks:
    raise SystemExit(f"LEAKAGE - a plant is in two sets: {dict(list(leaks.items())[:5])}")
changed = [r["file"] for r in old_rows
           if (prev_by_sha.get(r["src_sha256"]) or prev_by_file[r["file"].lower()])["split"] != r["split"]
           and "label_conflict" not in r["flags"]]
if changed:
    raise SystemExit(f"{len(changed)} old photos changed set - this must not happen: {changed[:5]}")

SPLIT_CSV = OUT / f"split_{VERSION}.csv"
with open(SPLIT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split", "src_sha256"], extrasaction="ignore")
    w.writeheader(); w.writerows(rows)

tally = defaultdict(Counter)
plants = defaultdict(set)
for r in rows:
    tally[r["split"]][r["label"]] += 1
    plants[(r["split"], r["label"])].add(r["group"])
print(f"new plants: {len(groups)}  (placed next to an old look-alike: {n_pinned}, excluded as ambiguous: {n_conflict})\n")
print(f"{'set':9s} {'healthy photos (plants)':>26s} {'unhealthy photos (plants)':>28s}")
for s in ("train", "val", "test", "excluded"):
    h, u = tally[s]["Healthy"], tally[s]["Unhealthy"]
    print(f"{s:9s} {h:>14d} ({len(plants[(s, 'Healthy')]):>3d})      {u:>16d} ({len(plants[(s, 'Unhealthy')]):>3d})")
for s in ("train", "val", "test"):
    for lab in ("Healthy", "Unhealthy"):
        if tally[s][lab] == 0:
            raise SystemExit(f"The {s} set has no {lab} photos - not enough distinct {lab} plants to split.")
print("\nLEAKAGE CHECK OK - no plant is in two sets; old photos kept their set.")
n_hv = tally["val"]["Healthy"]
if n_hv < MIN_HEALTHY_VAL_FOR_AUTO_THRESHOLDS:
    print(f"\nNOTE: only {n_hv} healthy validation photos. Thresholds will stay at the hand-set 0.50 / 0.90 "
          f"until there are at least {MIN_HEALTHY_VAL_FOR_AUTO_THRESHOLDS}.")

## Step 4 — Extra crops of healthy training photos

While healthy photos are scarce, each healthy **training** photo is also cut into 7
overlapping crops (same as v1). Once there are enough healthy photos this switches off
by itself, because crops make healthy photos look more zoomed-in than infected ones.

In [ ]:
train_healthy = [r for r in rows if r["split"] == "train" and r["label"] == "Healthy"]
do_tiles = (len(train_healthy) < TILE_AUTO_BELOW) if TILE_HEALTHY == "auto" else bool(TILE_HEALTHY)
tile_rows = []
if do_tiles:
    (TILES / "Healthy").mkdir(parents=True, exist_ok=True)
    for r in train_healthy:
        img = ImageOps.exif_transpose(Image.open(r["orig"])).convert("RGB")
        w_, h_ = img.size
        side = int(min(w_, h_) * 0.55)
        xs = [int(i * (w_ - side) / 2) for i in range(3)]
        ys = [int(j * (h_ - side) / 1) for j in range(2)]
        boxes = [(x, y) for y in ys for x in xs] + [((w_ - side) // 2, (h_ - side) // 2)]
        for k, (x, y) in enumerate(boxes):
            name = f"Healthy/{Path(r['file']).stem}_tile{k}.jpg"
            shrink(img.crop((x, y, x + side, y + side))).save(TILES / name, "JPEG", quality=90)
            tile_rows.append({"file": name, "label": "Healthy", "group": r["group"], "split": "train", "tile": True})
print(f"healthy training photos: {len(train_healthy)}  ->  tiling {'ON' if do_tiles else 'OFF'}, {len(tile_rows)} crops added")

## Step 5 — Train contender B

Same recipe as v1: ConvNeXt-Tiny at 384 px, 2 warm-up epochs on the head, then the whole
network at lr 1e-4, AdamW, balanced sampling, class-weighted loss, early stopping.

One change from v1: the best epoch is chosen by validation PR-AUC **and, on a tie, by
the lower balanced validation loss**. In v1, PR-AUC hit 1.0 early and training stopped
on the first epoch that reached it; the tie-break keeps improving after that point.

In [ ]:
def set_seed():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.10),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([   # MUST stay identical in predict_folder.py and the API
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CwsbDataset(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open((TILES if r.get("tile") else CLEAN) / r["file"]).convert("RGB")
        return self.tf(img), 1 if r["label"] == "Unhealthy" else 0


N_WORKERS = 2 if DEVICE == "cuda" else 0


@torch.no_grad()
def predict(model, rws):
    """-> (labels, P(infected), balanced loss) for a list of rows."""
    model.eval()
    ys, ps = [], []
    for x, y in DataLoader(CwsbDataset(rws, eval_tf), batch_size=32, num_workers=N_WORKERS):
        ps += torch.softmax(model(x.to(DEVICE)).float(), 1)[:, 1].cpu().tolist()
        ys += y.tolist()
    ys, ps = np.array(ys), np.clip(np.array(ps), 1e-6, 1 - 1e-6)
    nll = -(ys * np.log(ps) + (1 - ys) * np.log(1 - ps))
    bal = float(np.mean([nll[ys == c].mean() for c in (0, 1) if (ys == c).any()]))
    return ys, ps, bal


def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = timm.create_model(ck.get("backbone", BACKBONE), pretrained=False, num_classes=2)
    m.load_state_dict(ck["state_dict"])
    return m.to(DEVICE).eval()


set_seed()
tr = [r for r in rows if r["split"] == "train"] + tile_rows
va = [r for r in rows if r["split"] == "val"]
te = [r for r in rows if r["split"] == "test"]
n_pos = sum(r["label"] == "Unhealthy" for r in tr)
n_neg = len(tr) - n_pos
print(f"train {len(tr)} ({n_neg} healthy incl. crops, {n_pos} unhealthy)   val {len(va)}   test {len(te)}")

if START_FROM == "previous":
    model = load_model(PREV_MODEL_PATH)
    warmup = 0
    print(f"starting from the previous model: {PREV_MODEL_PATH}")
else:
    model = timm.create_model(BACKBONE, pretrained=True, num_classes=2).to(DEVICE)
    warmup = 2
    print("starting from ImageNet weights")

weights = [1.0 / (n_pos if r["label"] == "Unhealthy" else n_neg) for r in tr]
dl = DataLoader(CwsbDataset(tr, train_tf), batch_size=BATCH, num_workers=N_WORKERS,
                sampler=WeightedRandomSampler(weights, num_samples=len(tr), replacement=True))
crit = nn.CrossEntropyLoss(weight=torch.tensor([len(tr) / (2 * n_neg), len(tr) / (2 * n_pos)],
                                               dtype=torch.float32).to(DEVICE))
head = [p for n, p in model.named_parameters() if "head" in n]
body = [p for n, p in model.named_parameters() if "head" not in n]

best_key, best_ep, since, log = None, -1, 0, []
t0 = time.time()
for ep in range(EPOCHS):
    warm = ep < warmup
    for p in body:
        p.requires_grad = not warm
    opt = torch.optim.AdamW([{"params": head, "lr": 1e-3 if warm else 1e-4},
                             {"params": body, "lr": 0 if warm else 1e-4}], weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(dl))
    model.train()
    tot = 0.0
    for x, y in dl:
        opt.zero_grad()
        loss = crit(model(x.to(DEVICE)), y.to(DEVICE))
        loss.backward()
        opt.step()
        sched.step()
        tot += loss.item()
    yva, pva, vloss = predict(model, va)
    ap = average_precision_score(yva, pva)
    key = (round(ap, 4), -round(vloss, 4))
    better = best_key is None or key > best_key
    log.append({"epoch": ep, "phase": "warm" if warm else "full", "train_loss": round(tot / len(dl), 4),
                "val_pr_auc": round(ap, 4), "val_balanced_loss": round(vloss, 4), "saved": int(better)})
    print(f"ep{ep:02d} {'warm' if warm else 'full'}: train loss {tot / len(dl):.4f}   val PR-AUC {ap:.4f}   "
          f"val balanced loss {vloss:.4f}   {'<- best so far' if better else ''}   ({time.time() - t0:.0f}s)")
    if better:
        best_key, best_ep, since = key, ep, 0
        torch.save({"mode": "B", "backbone": BACKBONE, "img": IMG, "state_dict": model.state_dict(),
                    "val_pr_auc": ap, "val_balanced_loss": vloss, "epoch": ep, "version": VERSION},
                   OUT / "contender_B.pt")
    else:
        since += 1
        if since >= PATIENCE:
            print(f"early stop: best was epoch {best_ep}")
            break

with open(OUT / "training_log.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(log[0].keys()))
    w.writeheader(); w.writerows(log)
print(f"\nsaved {OUT / 'contender_B.pt'}  (epoch {best_ep}, val PR-AUC {best_key[0]}, val balanced loss {-best_key[1]})")

## Step 6 — Thresholds, from the healthy validation scores

- `T_low`: 95% of healthy validation photos score below it → *no visible signs*.
- `T_high`: at least 0.90, and above the healthy scores → *WSB likely*.
- In between → *inconclusive, retake the photo*.

The thresholds are **never** taken from infected-score quantiles (that was the v1 bug
that collapsed the band to 0.008). The band is forced to stay at least 0.10 wide.
With fewer than 20 healthy validation photos the hand-set 0.50 / 0.90 are kept.

In [ ]:
model = load_model(OUT / "contender_B.pt")
yva, pva, _ = predict(model, va)
yte, pte, _ = predict(model, te)
for r, p in zip(va, pva):
    r["score"] = float(p)
for r, p in zip(te, pte):
    r["score"] = float(p)

h_val = np.sort(pva[yva == 0])
if len(h_val) >= MIN_HEALTHY_VAL_FOR_AUTO_THRESHOLDS:
    T_LOW = round(float(np.clip(np.quantile(h_val, 0.95), 0.20, 0.80)), 2)
    T_HIGH = round(float(np.clip(max(0.90, np.quantile(h_val, 0.99) + 0.05), T_LOW + 0.10, 0.98)), 2)
    how = (f"derived from {len(h_val)} healthy validation photos: T_low = their 95th percentile, "
           f"T_high = max(0.90, 99th percentile + 0.05)")
else:
    T_LOW, T_HIGH = 0.50, 0.90
    how = (f"hand-set 0.50 / 0.90 (v1 values): only {len(h_val)} healthy validation photos, "
           f"too few to derive thresholds from")
assert T_HIGH - T_LOW >= 0.10, "threshold band collapsed - do not use this model"
print(f"T_low = {T_LOW}   T_high = {T_HIGH}\n  {how}")
print(f"  healthy validation scores: min {h_val.min():.3f}  median {np.median(h_val):.3f}  max {h_val.max():.3f}")
print(f"  infected validation scores: min {pva[yva == 1].min():.3f}  median {np.median(pva[yva == 1]):.3f}")

## Step 7 — Honest test on held-out plants, and comparison with the previous model

In [ ]:
def verdict(s, lo, hi):
    return "wsb_likely" if s >= hi else ("no_visible_signs" if s < lo else "inconclusive")


def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / d
    return (max(0.0, c - h), min(1.0, c + h))


def tally_items(items, lo, hi):
    tp = fn = fp = tn = inc_i = inc_h = 0
    for s, infected in items:
        v = verdict(s, lo, hi)
        if v == "inconclusive":
            inc_i += infected; inc_h += not infected
        elif v == "wsb_likely":
            tp += infected; fp += not infected
        else:
            fn += infected; tn += not infected
    return {"caught": tp, "missed": fn, "false_alarms": fp, "cleared": tn,
            "inconclusive": inc_i + inc_h, "inconclusive_infected": inc_i, "inconclusive_healthy": inc_h}


def summarise(rws, scores, lo, hi, by_plant):
    if by_plant:
        best = {}
        for r, s in zip(rws, scores):
            if s > best.get(r["group"], (-1, None))[0]:
                best[r["group"]] = (s, r["label"] == "Unhealthy")
        items = list(best.values())
    else:
        items = [(s, r["label"] == "Unhealthy") for r, s in zip(rws, scores)]
    return tally_items(items, lo, hi)


def show(title, t, unit):
    n_inf = t["caught"] + t["missed"] + t["inconclusive_infected"]
    n_hea = t["cleared"] + t["false_alarms"] + t["inconclusive_healthy"]
    print(f"\n{title}  ({n_inf} infected + {n_hea} healthy {unit})")
    lo, hi = wilson(t["caught"], n_inf)
    print(f"  infected caught    : {t['caught']:4d} / {n_inf}  = {t['caught'] / max(n_inf, 1):.1%}   (95% range {lo:.0%} - {hi:.0%})")
    print(f"  infected MISSED    : {t['missed']:4d}   (called 'no visible signs')")
    lo, hi = wilson(t["cleared"], n_hea)
    print(f"  healthy cleared    : {t['cleared']:4d} / {n_hea}  = {t['cleared'] / max(n_hea, 1):.1%}   (95% range {lo:.0%} - {hi:.0%})")
    print(f"  healthy FALSE ALARM: {t['false_alarms']:4d}   (called 'WSB likely')")
    print(f"  inconclusive       : {t['inconclusive']:4d}   ({t['inconclusive_infected']} infected, {t['inconclusive_healthy']} healthy) -> retake")


print(f"=== MODEL {VERSION}   thresholds {T_LOW} / {T_HIGH} ===")
print("(scored on the 768 px training copies, like the v1 'test2' figures)")
val_photo = summarise(va, pva, T_LOW, T_HIGH, False)
test_photo = summarise(te, pte, T_LOW, T_HIGH, False)
test_plant = summarise(te, pte, T_LOW, T_HIGH, True)
show("VALIDATION, per photo (used to pick the model and thresholds)", val_photo, "photos")
show("TEST, per photo (held out - the honest number)", test_photo, "photos")
show("TEST, per plant (how the app decides: highest photo score)", test_plant, "plants")
test_auc = float(roc_auc_score(yte, pte))
print(f"\nTEST ROC-AUC (threshold-free separation of healthy vs infected): {test_auc:.3f}")

comparison = None
if PREV_MODEL_PATH:
    prev_cfg_path = PREV_MODEL_PATH.parent / "config.json"
    p_lo, p_hi = 0.50, 0.90
    if prev_cfg_path.exists():
        pc = json.loads(prev_cfg_path.read_text())
        p_lo, p_hi = pc.get("T_low", p_lo), pc.get("T_high", p_hi)
    prev_model = load_model(PREV_MODEL_PATH)
    _, ppte, _ = predict(prev_model, te)
    del prev_model
    old_te = [i for i, r in enumerate(te) if r["old"]]
    comparison = {}
    print(f"\n\n=== {VERSION} versus the previous model, same test photos ===")
    print(f"(previous model uses its own thresholds {p_lo} / {p_hi})")
    for name, idx in (("all test photos", list(range(len(te)))), ("only the OLD test photos", old_te)):
        if not idx:
            continue
        sub = [te[i] for i in idx]
        a = summarise(sub, [ppte[i] for i in idx], p_lo, p_hi, False)
        b = summarise(sub, [pte[i] for i in idx], T_LOW, T_HIGH, False)
        comparison[name] = {"previous": a, VERSION: b}
        print(f"\n  {name} ({len(idx)}):")
        print(f"    {'':22s}{'previous':>10s}{VERSION:>10s}")
        for k, lab in (("caught", "infected caught"), ("missed", "infected missed"), ("cleared", "healthy cleared"),
                       ("false_alarms", "healthy false alarm"), ("inconclusive", "inconclusive")):
            print(f"    {lab:22s}{a[k]:>10d}{b[k]:>10d}")
        if len({r['label'] for r in sub}) == 2:
            ys = [r["label"] == "Unhealthy" for r in sub]
            print(f"    {'ROC-AUC':22s}{roc_auc_score(ys, [ppte[i] for i in idx]):>10.3f}{roc_auc_score(ys, [pte[i] for i in idx]):>10.3f}")
    for r, p in zip(te, ppte):
        r["previous_score"] = float(p)

# worst mistakes, for a human to look at
wrong = sorted((r for r in te if verdict(r["score"], T_LOW, T_HIGH) != "inconclusive"
                and (verdict(r["score"], T_LOW, T_HIGH) == "wsb_likely") != (r["label"] == "Unhealthy")),
               key=lambda r: abs(r["score"] - 0.5), reverse=True)
print(f"\n\nTest photos the model got WRONG ({len(wrong)}):")
for r in wrong[:25]:
    print(f"  {r['file']:55s} {r['group']:11s} truth {r['label']:9s} score {r['score']:.3f}")

In [ ]:
# ---------- Save everything ----------
with open(OUT / "scores_val_test.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split", "score", "verdict", "previous_score"],
                       extrasaction="ignore")
    w.writeheader()
    for r in va + te:
        w.writerow(dict(r, score=round(r["score"], 4), verdict=verdict(r["score"], T_LOW, T_HIGH),
                        previous_score=round(r["previous_score"], 4) if "previous_score" in r else ""))

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
    for ax, (name, ys, ps) in zip(axes, (("validation", yva, pva), ("test", yte, pte))):
        bins = np.linspace(0, 1, 26)
        ax.hist(ps[ys == 1], bins=bins, color="#c0392b", alpha=0.75, label=f"infected ({int((ys == 1).sum())})")
        ax.hist(ps[ys == 0], bins=bins, color="#1f77b4", alpha=0.85, label=f"healthy ({int((ys == 0).sum())})")
        ax.axvline(T_LOW, color="k", ls="--", lw=1); ax.axvline(T_HIGH, color="k", ls="--", lw=1)
        ax.set_yscale("log"); ax.set_title(f"{VERSION} scores - {name}"); ax.set_xlabel("score (0 = healthy, 1 = infected)")
        ax.set_ylabel("photos (log scale)"); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(OUT / "score_histograms.png", dpi=130)
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        pass
    plt.close(fig)
except Exception as e:
    print("plot skipped:", e)

counts = {s: {lab: tally[s][lab] for lab in ("Healthy", "Unhealthy")} for s in ("train", "val", "test", "excluded")}
plant_counts = {s: {lab: len(plants[(s, lab)]) for lab in ("Healthy", "Unhealthy")} for s in ("train", "val", "test")}
(OUT / "config.json").write_text(json.dumps({
    "version": VERSION,
    "model": "contender_B.pt",
    "backbone": BACKBONE,
    "img": IMG,
    "T_low": T_LOW,
    "T_high": T_HIGH,
    "plant_score": "max over the plant's photos",
    "answers": {"score >= T_high": "wsb_likely", "T_low <= score < T_high": "inconclusive",
                "score < T_low": "no_visible_signs"},
    "calibrated_on": "validation split only; test split used as a check",
    "rationale": how,
    "trained_on": {"data_sources": [Path(s).name for s in DATA_SOURCES], "start_from": START_FROM,
                   "previous_split": PREVIOUS_SPLIT, "previous_model": PREVIOUS_MODEL,
                   "photos": counts, "plants": plant_counts, "healthy_tiles": len(tile_rows),
                   "best_epoch": best_ep, "date": datetime.now().strftime("%Y-%m-%d")},
    "scored_at": f"short side {TRAIN_SHORT_SIDE} px copies",
    "val_photo_level": val_photo,
    "test_photo_level": test_photo,
    "test_plant_level": test_plant,
    "test_roc_auc": round(test_auc, 4),
    "comparison_with_previous": comparison,
}, indent=2))
(OUT / "training_result.json").write_text(json.dumps(
    {"val_pr_auc": {"B": best_key[0]}, "val_balanced_loss": {"B": -best_key[1]}, "winner": "B",
     "backbone": BACKBONE, "img": IMG, "seed": SEED, "version": VERSION}, indent=2))

print(f"Saved to {OUT}:")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / 1e6:8.2f} MB")
print(f"""
NEXT, on the laptop:
  1. Download the Drive folder  {OUT.name}/  into the repo as  models/{VERSION}/
     (contender_B.pt must sit directly inside models/{VERSION}/).
  2. Copy  split_{VERSION}.csv  into the repo's  splits/  folder and commit it.
     The NEXT retrain must use it:  PREVIOUS_SPLIT = "split_{VERSION}.csv"
                                    PREVIOUS_MODEL = "{VERSION}/contender_B.pt"
  3. Record the numbers above in PROJECT_STATE.md and your working log.
""")